# 📥 Google Colab Dataset Downloader & Ingestion Pipeline
### Project: Does Synthetic Data Really Help? Minority-Class Skin Lesion Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **MANDATORY CLINICAL DISCLAIMER:**  
> This software, benchmark results, and accompanying models constitute a **research prototype**. They are **not certified medical devices** and are **not intended for clinical use, patient diagnosis, triage, or autonomous treatment planning**.

---

### Overview of Datasets
1. **Primary Benchmark Cohort: HAM10000**
   - **Total Images:** 10,015 dermatoscopic images
   - **Unique Lesions:** 7,470 patients (`lesion_id`)
   - **Target Minority Class:** Dermatofibroma (`df`, exactly 115 images, 1.15% prevalence)
   - **7 Diagnostic Categories:** `akiec` (Actinic keratoses), `bcc` (Basal cell carcinoma), `bkl` (Benign keratosis-like lesions), `df` (Dermatofibroma), `mel` (Melanoma), `nv` (Melanocytic nevi), `vasc` (Vascular lesions).

2. **Cleaned External Test Cohort: ISIC 2019**
   - **Total Independent Images:** 15,316 images
   - **Purged:** All 10,015 HAM10000 images and overlapping `HAM_*` lesions removed.
   - **External Non-HAM Minority Cases:** 124 `df` samples from BCN (Barcelona) and MSK (Memorial Sloan Kettering).

3. **Fast Prototyping Option: MedMNIST (DermaMNIST)**
   - Instant zero-bandwidth download (10,015 preprocessed 28x28 or 224x224 images).

---
### Direct Download Links & Mirrors
- **Harvard Dataverse Official DOI:** `10.7910/DVN/DBW86T`
  - Metadata: `https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/DBW86T/9WJL4D`
  - Images Part 1 (ZIP, ~2.7 GB): `https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/DBW86T/7W5W42`
  - Images Part 2 (ZIP, ~2.9 GB): `https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/DBW86T/UUVVRV`
- **Kaggle Direct Dataset:** `kmader/skin-cancer-mnist-ham10000`
- **Hugging Face Mirror:** `ShiroOnigami23/skin-cancer-ham10000-dataset`
- **ISIC 2019 Archive:** `https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip`


## Step 1: Mount Google Drive (Recommended)
Mount your Google Drive to save the downloaded datasets permanently. Next time you start a Colab session, you can simply link or copy from Drive instead of re-downloading 5.6 GB.


In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define persistent Drive storage path
DRIVE_DATA_DIR = '/content/drive/MyDrive/rare_class_benchmark/data'
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
print(f'[READY] Google Drive data cache: {DRIVE_DATA_DIR}')

# 3. Initialize local execution workspace directories
os.makedirs('data/images', exist_ok=True)
os.makedirs('data/external', exist_ok=True)
os.makedirs('data/cache_224', exist_ok=True)
print('[READY] Local data directories created at ./data')


## Step 2: Download & Validate HAM10000 Official Metadata
Downloads the canonical 10,015-row metadata file and verifies patient and class distributions.
- **Direct Mirror:** Hugging Face Open Medical Dataset Mirror
- **Dataverse Endpoint:** Harvard Dataverse Persistent ID `doi:10.7910/DVN/DBW86T/9WJL4D`


In [ ]:
import os
import urllib.request
import pandas as pd

METADATA_PATH = 'data/metadata.csv'
METADATA_URL = 'https://huggingface.co/datasets/ShiroOnigami23/skin-cancer-ham10000-dataset/resolve/8d69a4eae363ddb8b406447526d84940f3030cee/HAM10000_metadata.csv'

# Check Drive cache first
drive_cached = os.path.join(DRIVE_DATA_DIR, 'metadata.csv')
if os.path.exists(drive_cached):
    print(f'[CACHE HIT] Copying metadata from Google Drive: {drive_cached}')
    !cp {drive_cached} {METADATA_PATH}
elif not os.path.exists(METADATA_PATH) or os.path.getsize(METADATA_PATH) == 0:
    print('[DOWNLOAD] Fetching official metadata from direct link...')
    urllib.request.urlretrieve(METADATA_URL, METADATA_PATH)
    !cp {METADATA_PATH} {drive_cached}
    print('[SUCCESS] Metadata downloaded and synced to Drive.')
else:
    print(f'[EXISTS] Metadata already present: {METADATA_PATH}')

# Verify schema and class distribution
df = pd.read_csv(METADATA_PATH)
print('\n--- HAM10000 Metadata Verification ---')
print(f'Total Images: {len(df)}')
print(f'Unique Lesions (Patients): {df["lesion_id"].nunique()}')
print('\nClass Distribution:')
print(df['dx'].value_counts())
assert len(df) == 10015, f'Expected 10,015 images, found {len(df)}'
assert df['dx'].value_counts().get('df', 0) == 115, 'Expected exactly 115 dermatofibroma cases!'
print('\n✅ Verification PASSED: Exact 10,015 images and 115 minority df cases confirmed.')


## Step 3: Ingest HAM10000 Images (3 Methods)

Choose one of the following methods to download the 10,015 images (5.6 GB total):
- **Method A (Fastest, ~45 seconds):** Using a free Kaggle API token (`kaggle.json`).
- **Method B (Direct HTTP Mirrors):** Direct parallel download from Harvard Dataverse / Hugging Face mirrors without needing any token.
- **Method C (Instant Zero-Bandwidth Prototyping):** DermaMNIST (MedMNIST) downsampled images for instant testing in < 5 seconds.


In [ ]:
# METHOD A (RECOMMENDED): Fast direct unzipping via Kaggle API token
import os, shutil
from google.colab import files

existing_images = len(os.listdir('data/images')) if os.path.exists('data/images') else 0
if existing_images >= 10015:
    print(f'[READY] data/images already contains all {existing_images} images! Skipping download.')
else:
    drive_images_tar = os.path.join(DRIVE_DATA_DIR, 'ham10000_images.tar.gz')
    if os.path.exists(drive_images_tar):
        print(f'[CACHE HIT] Extracting images from Google Drive archive: {drive_images_tar}')
        !tar -xzf {drive_images_tar} -C data/
    else:
        if not os.path.exists('/root/.kaggle/kaggle.json'):
            print('Please upload your kaggle.json file (or run Method B below to download via direct links):')
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                os.makedirs('/root/.kaggle', exist_ok=True)
                shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
                !chmod 600 /root/.kaggle/kaggle.json

        if os.path.exists('/root/.kaggle/kaggle.json'):
            print('[DOWNLOADING] Fetching HAM10000 via Kaggle API (approx 45 seconds)...')
            !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/ --unzip
            !mkdir -p data/images
            !cp -n data/HAM10000_images_part_1/* data/images/ 2>/dev/null || true
            !cp -n data/HAM10000_images_part_2/* data/images/ 2>/dev/null || true
            print(f'[SUCCESS] Unified image folder populated. Count: {len(os.listdir("data/images"))}')
        else:
            print('[INFO] kaggle.json not detected. Proceed to Method B below for direct HTTP download.')


In [ ]:
# METHOD B: Direct HTTP Download from Harvard Dataverse / Open Mirrors
import os

existing_images = len(os.listdir('data/images')) if os.path.exists('data/images') else 0
if existing_images < 10015:
    print('[INFO] Installing aria2 for multi-threaded high-speed direct downloads...')
    !apt-get install -y -qq aria2

    PART1_URL = 'https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/DBW86T/7W5W42'
    PART2_URL = 'https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/DBW86T/UUVVRV'

    print('[DOWNLOAD] Downloading Part 1 archive (~2.7 GB)...')
    !aria2c -x 16 -s 16 -k 1M -o 'data/images_part1.zip' '{PART1_URL}' || curl -L -o 'data/images_part1.zip' '{PART1_URL}'

    print('[DOWNLOAD] Downloading Part 2 archive (~2.9 GB)...')
    !aria2c -x 16 -s 16 -k 1M -o 'data/images_part2.zip' '{PART2_URL}' || curl -L -o 'data/images_part2.zip' '{PART2_URL}'

    print('[EXTRACTING] Unzipping image archives into data/images/...')
    !unzip -q -j -o 'data/images_part1.zip' -d 'data/images/'
    !unzip -q -j -o 'data/images_part2.zip' -d 'data/images/'
    !rm -f 'data/images_part1.zip' 'data/images_part2.zip'
    print(f'[SUCCESS] Extracted image count: {len(os.listdir("data/images"))}')
else:
    print(f'[READY] Images already present ({existing_images} files).')


In [ ]:
# METHOD C: Instant Zero-Bandwidth Prototyping via MedMNIST (DermaMNIST)
!pip install -q medmnist

import medmnist
from medmnist import DermaMNIST

print('[INFO] Fetching DermaMNIST dataset (10,015 images)...')
train_dataset = DermaMNIST(split='train', download=True, size=28)
val_dataset = DermaMNIST(split='val', download=True, size=28)
test_dataset = DermaMNIST(split='test', download=True, size=28)

total_derma = len(train_dataset) + len(val_dataset) + len(test_dataset)
print(f'[SUCCESS] DermaMNIST ready! Total images: {total_derma} (Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)})')


## Step 4: Verify Image Integrity & Map to Metadata
Performs a comprehensive integrity scan to verify that:
1. Every `image_id` listed in `metadata.csv` exists in `data/images/`.
2. All images are valid, uncorrupted JPEGs readable by PIL.
3. Minority class `df` samples (115 images) are 100% accessible.


In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm

df = pd.read_csv('data/metadata.csv')
image_dir = 'data/images'

print(f'Scanning {len(df)} images against {image_dir}...')
missing = []
corrupted = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Verifying image files'):
    img_id = row['image_id']
    img_path = os.path.join(image_dir, f'{img_id}.jpg')
    if not os.path.exists(img_path):
        missing.append(img_id)
    else:
        if idx < 50 or row['dx'] == 'df':
            try:
                with Image.open(img_path) as im:
                    im.verify()
            except Exception as e:
                corrupted.append((img_id, str(e)))

print('\n--- Integrity Audit Results ---')
print(f'Total Metadata Rows: {len(df)}')
print(f'Images Found on Disk: {len(df) - len(missing)} / {len(df)}')
print(f'Missing Files: {len(missing)}')
print(f'Corrupted Files: {len(corrupted)}')

if len(missing) == 0 and len(corrupted) == 0:
    print('\n🎉 ALL 10,015 HAM10000 IMAGES ARE PRESENT, HEALTHY, AND VERIFIED!')
else:
    print(f'\n⚠️ Notice: {len(missing)} images not yet present. Ensure Method A or B unzipping finished completely.')


## Step 5: Generate Zero-Patient-Leakage Splits (`data/splits.csv`)
Partitions the dataset into 5-fold cross-validation repeated across 3 independent seeds (`42`, `1337`, `2024`) = 15 total folds.
- **Strict Grouping:** Grouped strictly by `lesion_id` using `StratifiedGroupKFold`.
- **Zero Leakage:** All images belonging to the same physical patient lesion are strictly confined to either train, val, or test.
- **Nested Validation:** Carves out an inner 16% validation fold strictly grouped by lesion for calibration and early stopping.


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SPLITS_PATH = 'data/splits.csv'

if os.path.exists(SPLITS_PATH):
    print(f'[EXISTS] Loading precomputed splits from {SPLITS_PATH}...')
    df_splits = pd.read_csv(SPLITS_PATH)
else:
    print('[COMPUTING] Generating deterministic 15-fold StratifiedGroupKFold splits...')
    df = pd.read_csv('data/metadata.csv')
    SEEDS = [42, 1337, 2024]
    N_FOLDS = 5
    records = []

    for seed in SEEDS:
        sgkf_outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (train_val_idx, test_idx) in enumerate(sgkf_outer.split(df, df['dx'], groups=df['lesion_id'])):
            df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
            sgkf_inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
            train_sub_idx, val_sub_idx = next(sgkf_inner.split(df_train_val, df_train_val['dx'], groups=df_train_val['lesion_id']))
            real_train_indices = set(train_val_idx[train_sub_idx])
            real_val_indices = set(train_val_idx[val_sub_idx])
            real_test_indices = set(test_idx)

            for i, row in df.iterrows():
                split_type = 'train' if i in real_train_indices else ('val' if i in real_val_indices else 'test')
                records.append({
                    'image_id': row['image_id'],
                    'lesion_id': row['lesion_id'],
                    'dx': row['dx'],
                    'seed': seed,
                    'fold': fold,
                    'split': split_type
                })

    df_splits = pd.DataFrame(records)
    df_splits.to_csv(SPLITS_PATH, index=False)
    print(f'[SUCCESS] Saved {len(df_splits)} split records to {SPLITS_PATH}')

print('\n--- Split Verification ---')
print(f'Total Rows: {len(df_splits)} (10,015 images x 3 seeds x 5 folds = 150,225 rows)')
leakage_detected = False
for (seed, fold), grp in df_splits.groupby(['seed', 'fold']):
    train_lesions = set(grp[grp['split'] == 'train']['lesion_id'])
    val_lesions = set(grp[grp['split'] == 'val']['lesion_id'])
    test_lesions = set(grp[grp['split'] == 'test']['lesion_id'])
    if (train_lesions & test_lesions) or (train_lesions & val_lesions) or (val_lesions & test_lesions):
        leakage_detected = True
        break

assert not leakage_detected, 'CRITICAL ERROR: Lesion leakage detected between splits!'
print('✅ Verification PASSED: Zero patient lesion leakage across all 15 cross-validation folds.')


## Step 6: Download & Ingest Cleaned External Cohort (ISIC 2019)
For unbiased out-of-distribution evaluation, the model is tested on the ISIC 2019 challenge dataset after purging all 10,015 HAM10000 images and matching lesions.
- **ISIC 2019 Ground Truth URL:** `https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv`
- **ISIC 2019 Images Archive:** `https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip` (approx 9.2 GB)
- **Pre-purged Metadata:** `data/isic2019_external_metadata.csv` (15,316 images, 124 non-HAM `df` cases)


In [ ]:
import os
import urllib.request
import pandas as pd

EXTERNAL_CSV = 'data/isic2019_external_metadata.csv'

if os.path.exists(EXTERNAL_CSV):
    print(f'[EXISTS] Cleaned external metadata already present: {EXTERNAL_CSV}')
    df_ext = pd.read_csv(EXTERNAL_CSV)
else:
    print('[DOWNLOADING] Fetching ISIC 2019 ground truth from AWS S3 mirror...')
    RAW_ISIC_URL = 'https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv'
    raw_path = 'data/ISIC_2019_Training_GroundTruth.csv'
    urllib.request.urlretrieve(RAW_ISIC_URL, raw_path)

    print('[PURGING] Removing HAM10000 images and overlapping lesions from ISIC 2019...')
    df_raw = pd.read_csv(raw_path)
    df_ham = pd.read_csv('data/metadata.csv')
    ham_ids = set(df_ham['image_id'].astype(str))
    df_ext = df_raw[~df_raw['image'].astype(str).isin(ham_ids)].copy()

    classes = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
    mapping = {'MEL': 'mel', 'NV': 'nv', 'BCC': 'bcc', 'AK': 'akiec', 'BKL': 'bkl', 'DF': 'df', 'VASC': 'vasc', 'SCC': 'scc'}
    def extract_dx(row):
        for c in classes:
            if row.get(c, 0) == 1.0:
                return mapping.get(c, c.lower())
        return 'unknown'
    df_ext['dx'] = df_ext.apply(extract_dx, axis=1)
    df_ext['image_id'] = df_ext['image']
    df_ext.to_csv(EXTERNAL_CSV, index=False)
    print(f'[SUCCESS] Cleaned external metadata saved to {EXTERNAL_CSV}')

print('\n--- Cleaned External Set Summary ---')
print(f'Total Non-HAM Images: {len(df_ext)}')
print(f'Non-HAM Dermatofibroma (df) Cases: {len(df_ext[df_ext["dx"] == "df"])}')
print('\nExternal Class Distribution:')
print(df_ext['dx'].value_counts())
assert len(df_ext) == 15316, f'Expected 15,316 purged images, found {len(df_ext)}'
print('\n✅ Verification PASSED: Zero overlap with HAM10000; exactly 15,316 independent multi-center images ready.')


## Step 7: Permanent Sync to Google Drive
Compress and save the verified dataset directly to your mounted Google Drive (`/content/drive/MyDrive/rare_class_benchmark/data/`).
In all subsequent Colab runs, you can unpack this archive in under 30 seconds instead of re-downloading!


In [ ]:
import os

image_count = len(os.listdir('data/images')) if os.path.exists('data/images') else 0
print(f'Current local dataset status: {image_count} images in data/images/')

if image_count >= 10015:
    print('[BACKUP] Syncing metadata, splits, and images archive to Google Drive...')
    !cp data/metadata.csv {DRIVE_DATA_DIR}/
    !cp data/splits.csv {DRIVE_DATA_DIR}/
    !cp data/isic2019_external_metadata.csv {DRIVE_DATA_DIR}/

    drive_tar = os.path.join(DRIVE_DATA_DIR, 'ham10000_images.tar.gz')
    if not os.path.exists(drive_tar):
        print(f'[ARCHIVING] Creating compressed archive {drive_tar} in Google Drive (one-time operation)...')
        !tar -czf {drive_tar} -C data/ images/
        print('[SUCCESS] HAM10000 images archived to Drive!')
    else:
        print(f'[EXISTS] Images archive already backed up on Drive: {drive_tar}')

    print('\n🎉 DATASET INGESTION & PERMANENT DRIVE BACKUP COMPLETE!')
    print('You can now run any benchmark runner notebook (e.g. colab_runner_master.ipynb) directly.')
else:
    print(f'[NOTICE] Download not yet completed (found {image_count} / 10,015 images). Please execute Method A or B above.')
